# Módulo 3 - Ejercicio 1 (Solución Completa)
## Conexión a IBM Quantum, Primitives, Gestión de Cuota y Despacho Temprano a la QPU

- Tiempo en cola: Gratuito.
- Tiempo QPU por trabajo (1024 disparos): O(1) s
- Capacidad mensual: O(10^2) ejecuciones controladas.


In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.primitives import StatevectorSampler

print("Entorno Qiskit listo.")

Entorno Qiskit listo.


In [3]:
qc_ghz = QuantumCircuit(3, name="GHZ_3Q")
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(1, 2)
qc_ghz.measure_all()

print(qc_ghz.draw(output='text'))

        ┌───┐           ░ ┌─┐      
   q_0: ┤ H ├──■────────░─┤M├──────
        └───┘┌─┴─┐      ░ └╥┘┌─┐   
   q_1: ─────┤ X ├──■───░──╫─┤M├───
             └───┘┌─┴─┐ ░  ║ └╥┘┌─┐
   q_2: ──────────┤ X ├─░──╫──╫─┤M├
                  └───┘ ░  ║  ║ └╥┘
meas: 3/═══════════════════╩══╩══╩═
                           0  1  2 


In [ ]:
sampler_local = StatevectorSampler()
job_local = sampler_local.run([qc_ghz], shots=2048)
conteos_locales = job_local.result()[0].data.meas.get_counts()

print("Conteos ideales locales en estado GHZ:", conteos_locales)

In [ ]:
try:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    service = QiskitRuntimeService(channel="ibm_cloud")
    backend_qpu = service.least_busy(operational=True, simulator=False, min_num_qubits=3)
    print(f"QPU asignada: {backend_qpu.name}")
    qc_transpilado = transpile(qc_ghz, backend=backend_qpu, optimization_level=3)
    sampler_qpu = Sampler(mode=backend_qpu)
    job_qpu = sampler_qpu.run([qc_transpilado], shots=1024)
    id_trabajo = job_qpu.job_id()
    print(f" Trabajo despachado a la cola. Job ID: {id_trabajo}")
except Exception as e:
    print(f"Aviso de credenciales (normal si no hay token configurado en esta máquina): {e}")
    id_trabajo = "DEMO_JOB_LOCAL_12345"
    print(f"Job ID de referencia registrado: {id_trabajo}")